# The Azure Stay High Distribution Costs - Generate Data

## Import Libraries

In [1]:
import pandas as pd
import numpy as np
import datetime
import os
import random

In [2]:
!pip install xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 7.5 MB/s eta 0:00:00


## Configuration

In [3]:
# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

In [4]:
# 1. Setup Directory
output_dir = './Data'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

file_path = os.path.join(output_dir, 'AzureStay_ChannelProfitability.xlsx')

## Generate Data

In [5]:
# ==========================================
# 1. Generate dim_channels (Table 2)
# ==========================================
dim_channels_data = {
    'channel_id': ['CH_01', 'CH_02', 'CH_03', 'CH_04', 'CH_05'],
    'channel_name': ['Booking.com', 'Expedia', 'Direct Web', 'GDS', 'Walk-in'],
    'channel_type': ['OTA', 'OTA', 'Direct', 'Wholesale', 'Direct'],
    'commission_model': ['Percentage', 'Percentage', 'Flat Fee', 'Net Rate', 'None'],
    'default_commission_rate': [0.15, 0.18, 0.00, 0.10, 0.00], # Direct Web has 0 commission but marketing costs
    'contract_owner': ['Alice S.', 'Bob M.', 'Charlie D.', 'David W.', 'Eve Manager']
}
df_channels = pd.DataFrame(dim_channels_data)

# ==========================================
# 2. Generate dim_rate_codes (Table 3)
# ==========================================
dim_rate_codes_data = {
    'rate_code_id': ['RT_RACK', 'RT_CORP', 'RT_PROMO', 'RT_NET'],
    'rate_name': ['Rack Rate', 'Corporate Rate', 'Promotion Rate', 'Wholesale Net'],
    'is_commissionable': [True, True, True, False]
}
df_rate_codes = pd.DataFrame(dim_rate_codes_data)

# ==========================================
# 3. Generate fact_bookings (Table 1)
# ==========================================
num_bookings = 5000
booking_ids = [f"BKG-{str(i).zfill(5)}" for i in range(1, num_bookings + 1)]

# Generate Check-in Dates for year 2025
start_date = datetime.datetime(2025, 1, 1)
end_date = datetime.datetime(2025, 12, 31)
date_range = (end_date - start_date).days

check_in_dates = [start_date + datetime.timedelta(days=random.randint(0, date_range)) for _ in range(num_bookings)]
# Add random time for check-in (usually afternoon)
check_in_dates = [d.replace(hour=random.randint(14, 18), minute=random.randint(0, 59), second=random.randint(0, 59)) for d in check_in_dates]

# Generate Booking Dates (Lead time based on channel logic)
booking_dates = []
channels = []
rate_codes = []
gross_revenues = []
statuses = np.random.choice(['Checked-Out', 'Confirmed', 'Cancelled'], size=num_bookings, p=[0.75, 0.10, 0.15])

# Base prices
prices = {'RT_RACK': 5000, 'RT_CORP': 4000, 'RT_PROMO': 3500, 'RT_NET': 3000}

for i in range(num_bookings):
    cid = check_in_dates[i]
    is_weekend = cid.weekday() >= 4 # Friday, Saturday, Sunday
    
    # Logic to simulate Hypothesis 2 (OTAs strong on weekends, Direct/Corp on weekdays)
    if is_weekend:
        ch = np.random.choice(['CH_01', 'CH_02', 'CH_03', 'CH_04', 'CH_05'], p=[0.40, 0.35, 0.15, 0.05, 0.05])
    else:
        ch = np.random.choice(['CH_01', 'CH_02', 'CH_03', 'CH_04', 'CH_05'], p=[0.25, 0.20, 0.40, 0.10, 0.05])
    channels.append(ch)
    
    # Logic for Lead Time
    if ch == 'CH_05': # Walk-in
        lead_time = 0
    elif ch == 'CH_03': # Direct Web (often book slightly in advance)
        lead_time = random.randint(5, 30)
    else: # OTAs / GDS
        lead_time = random.randint(10, 90)
        
    b_date = cid - datetime.timedelta(days=lead_time)
    # Randomize booking time
    b_date = b_date.replace(hour=random.randint(8, 23), minute=random.randint(0, 59), second=random.randint(0, 59))
    booking_dates.append(b_date)
    
    # Logic for Rate Codes (Hypothesis 3)
    if ch in ['CH_01', 'CH_02']: # OTAs love Promos
        rc = np.random.choice(['RT_RACK', 'RT_PROMO'], p=[0.3, 0.7])
    elif ch == 'CH_03': # Direct gets Corp or Rack
        rc = np.random.choice(['RT_RACK', 'RT_CORP', 'RT_PROMO'], p=[0.5, 0.4, 0.1])
    elif ch == 'CH_04': # GDS gets Net
        rc = 'RT_NET'
    else: # Walk-in gets Rack
        rc = 'RT_RACK'
    rate_codes.append(rc)
    
    # Add a little randomness to the base price
    gross_revenues.append(prices[rc] * random.uniform(0.9, 1.1))

# Create Dataframe
df_bookings = pd.DataFrame({
    'booking_id': booking_ids,
    'booking_date': booking_dates,
    'check_in_date': check_in_dates,
    'channel_id': channels,
    'rate_code_id': rate_codes,
    'gross_room_revenue': gross_revenues,
    'status': statuses
})

# Calculate Commission and Net Revenue
df_bookings = df_bookings.merge(df_channels[['channel_id', 'default_commission_rate']], on='channel_id', how='left')
df_bookings = df_bookings.merge(df_rate_codes[['rate_code_id', 'is_commissionable']], on='rate_code_id', how='left')

# If rate is commissionable, calculate it. Else 0.
df_bookings['commission_amount'] = np.where(
    df_bookings['is_commissionable'] == True,
    df_bookings['gross_room_revenue'] * df_bookings['default_commission_rate'],
    0.0
)
df_bookings['net_room_revenue'] = df_bookings['gross_room_revenue'] - df_bookings['commission_amount']

# Drop merge helpers
df_bookings.drop(columns=['default_commission_rate', 'is_commissionable'], inplace=True)

# Format Datetimes exactly as YYYY-MM-DD HH:MM:SS
df_bookings['booking_date'] = df_bookings['booking_date'].dt.strftime('%Y-%m-%d %H:%M:%S')
df_bookings['check_in_date'] = df_bookings['check_in_date'].dt.strftime('%Y-%m-%d %H:%M:%S')

# Format floats
df_bookings['gross_room_revenue'] = df_bookings['gross_room_revenue'].round(2)
df_bookings['commission_amount'] = df_bookings['commission_amount'].round(2)
df_bookings['net_room_revenue'] = df_bookings['net_room_revenue'].round(2)

# ==========================================
# 5. Generate fact_marketing_spend (Table 4)
# ==========================================
# Simulate daily spend for Direct Web (CH_03)
spend_dates = pd.date_range(start='2025-01-01', end='2025-12-31', freq='D')
spend_ids = [f"SPD-{str(i).zfill(3)}" for i in range(1, len(spend_dates)*2 + 1)] # 2 platforms per day

mkt_data = []
s_idx = 0
for date in spend_dates:
    # Set time to end of day
    dt_str = date.replace(hour=23, minute=59, second=59).strftime('%Y-%m-%d %H:%M:%S')
    
    # Google Ads
    mkt_data.append({
        'spend_id': spend_ids[s_idx],
        'spend_date': dt_str,
        'channel_id': 'CH_03',
        'platform': 'Google Ads',
        'cost_amount': round(random.uniform(500, 1500), 2),
        'clicks': random.randint(50, 300)
    })
    s_idx += 1
    
    # Meta Ads
    mkt_data.append({
        'spend_id': spend_ids[s_idx],
        'spend_date': dt_str,
        'channel_id': 'CH_03',
        'platform': 'Meta Ads',
        'cost_amount': round(random.uniform(300, 1000), 2),
        'clicks': random.randint(100, 500)
    })
    s_idx += 1

df_spend = pd.DataFrame(mkt_data)

# ==========================================
# 6. Export to single Excel file with multiple sheets
# ==========================================
with pd.ExcelWriter(file_path, engine='xlsxwriter') as writer:
    df_bookings.to_excel(writer, sheet_name='fact_bookings', index=False)
    df_channels.to_excel(writer, sheet_name='dim_channels', index=False)
    df_rate_codes.to_excel(writer, sheet_name='dim_rate_codes', index=False)
    df_spend.to_excel(writer, sheet_name='fact_marketing_spend', index=False)

print(f"Data generated successfully! File saved to: {file_path}")

Data generated successfully! File saved to: ./Data/AzureStay_ChannelProfitability.xlsx
